# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guided tutorial for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source

The dataset metadata and structure are specified via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed in the current environment
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and prepare for access to its record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access the metadata (as an object, not as a dictionary!)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

# Display dataset citation
print(f"Cite as: {dataset.metadata.cite_as}")

## 2. Data Overview

List **all available record sets (@id)**, explore their associated fields and columns, **referencing all by their `@id`** as required for `mlcroissant`, and display their info for further exploration.

In [ ]:
# List all record sets in the dataset, referencing by their @id
print('Available record sets:')
record_sets = dataset.metadata.record_sets
for record_set in record_sets:
    print(f"  - @id: {record_set.id}\n    name: {record_set.name}\n    description: {record_set.description}")
    # List fields for the record set
    if hasattr(record_set, 'fields'):
        print('    Fields:')
        for field in record_set.fields:
            print(f"      - @id: {field.id} | name: {field.name} | data type: {field.data_type}")
    print()

## 3. Data Extraction

Load data from one or more record sets into DataFrames for interactive analysis. As an example, we select each record set by `@id`, and extract all available fields for exploration.

In [ ]:
# Collect all record set @id's
rs_ids = [rs.id for rs in dataset.metadata.record_sets]
print('Selected record sets:', rs_ids)

# Load records from each record set into a pandas DataFrame
dataframes = {}
for record_set_id in rs_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for record set '{record_set_id}': {df.columns.tolist()}")
    print(df.head(2))

# For further analysis, pick the first available record set as example
main_record_set_id = rs_ids[0]
print(f"\nMain record set chosen for analysis: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate common EDA steps:
- Filtering records
- Normalizing a numeric field
- Grouping and aggregation

> **Note:** All fields are referenced by their `@id` as shown above.

In [ ]:
# Choose a numeric field by its @id for demonstration
example_numeric_field_id = None
numeric_types = {'Float', 'Number', 'Integer'}
for f in dataset.metadata.record_sets[0].fields:
    if getattr(f, 'data_type', None) in numeric_types:
        example_numeric_field_id = f.id
        break
if not example_numeric_field_id:
    raise ValueError('No numeric field found in main record set.')
print(f"Chosen numeric field (@id): {example_numeric_field_id}")

df = dataframes[main_record_set_id]

# Clean: convert column to numeric if possible
df[example_numeric_field_id] = pd.to_numeric(df[example_numeric_field_id], errors='coerce')
threshold = df[example_numeric_field_id].mean() if df[example_numeric_field_id].notna().any() else 0
# Filter for above average values
filtered_df = df[df[example_numeric_field_id] > threshold].copy()
print(f"Number of filtered records with {example_numeric_field_id} > {threshold:.2f}: {len(filtered_df)}")

# Normalize the numeric field (z-score)
filtered_df[f"{example_numeric_field_id}_normalized"] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
print(filtered_df[[example_numeric_field_id, f"{example_numeric_field_id}_normalized"]].head())

# Attempt group by a categorical field (find the first Text field, if present)
example_group_field_id = None
for f in dataset.metadata.record_sets[0].fields:
    if getattr(f, 'data_type', None) == 'Text' and f.id != example_numeric_field_id:
        example_group_field_id = f.id
        break

if example_group_field_id and example_group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(example_group_field_id)[example_numeric_field_id].mean()
    print(f"\nGrouped mean {example_numeric_field_id} by field '{example_group_field_id}':")
    print(grouped_df.head())
else:
    print("No categorical field found for grouping.")

## 5. Visualization

Let's visualize the distribution of the chosen numeric field and, if a categorical field is available, display its grouped means.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram of the normalized numeric field
if not filtered_df.empty:
    plt.figure(figsize=(6, 4))
    filtered_df[f"{example_numeric_field_id}_normalized"].hist(bins=20)
    plt.title(f'Normalized distribution of {example_numeric_field_id}')
    plt.xlabel('Z-score')
    plt.ylabel('Count')
    plt.show()

    # If grouped means exist, bar plot of top 5 categories
    if example_group_field_id and example_group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(example_group_field_id)[example_numeric_field_id].mean().sort_values(ascending=False).head(5)
        if not grouped.empty:
            grouped.plot(kind='bar', figsize=(7, 4), title=f'Mean of {example_numeric_field_id} by {example_group_field_id}')
            plt.ylabel('Mean Value')
            plt.show()

## 6. Conclusion

- Successfully loaded and explored the FAIR^2 dataset Croissant package using `mlcroissant`.
- Demonstrated accessing fields by their `@id` in all extraction and EDA steps.
- Provided an example of visual EDA using filtered and normalized numeric values.

**Continue exploring by referencing additional record sets or fields by their `@id` as needed for your research!**